In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_cadastral")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-296e8f46-58b9-43e4-ae60-63a10c61ebbf;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 171ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [3]:
path = "s3a://bronze/base_dados_cadastrais/"
df_base_dados_cadastrais = spark.read.parquet(path)
df_base_dados_cadastrais.show(5, truncate=False)

25/12/29 08:05:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/12/29 08:05:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+----------+------+------+------+------+------+--------+------+------------+------------+------+----------+-------------------------+-------------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD |PROD|flag_mig2|STATUSRF|DATADENASCIMENTO|var_03|var_02|var_04|var_05|var_06|var_07|var_08|var_09|var_10|var_11|var_12    |var_13    |var_14|var_15|var_16|var_17|var_18|var_19  |var_20|var_21      |var_22      |var_23|var_24    |var_25                   |CEP_3_digitos|
+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+----------+------+------+------+------+------+--------+------+------------+------------+------+----------+-------------------------+-------------+
|77789989YZZ|202503|1              |0   |CMV |PRE      |REGULAR |09/09

In [4]:
df_base_dados_cadastrais.createOrReplaceTempView("raw_00")

In [5]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM raw_00
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5, truncate=False)

+------+------------+-------------+
|SAFRA |total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|653586      |653586       |
|202411|665737      |665737       |
|202412|646037      |646037       |
|202501|667227      |667227       |
|202502|619961      |619961       |
+------+------------+-------------+
only showing top 5 rows



In [6]:
print('lista de colunas para tipar')
for col in spark.table("raw_00").columns:
    print('cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
cast(NUM_CPF as) as NUM_CPF,
cast(SAFRA as) as SAFRA,
cast(FLAG_INSTALACAO as) as FLAG_INSTALACAO,
cast(FPD as) as FPD,
cast(PROD as) as PROD,
cast(flag_mig2 as) as flag_mig2,
cast(STATUSRF as) as STATUSRF,
cast(DATADENASCIMENTO as) as DATADENASCIMENTO,
cast(var_03 as) as var_03,
cast(var_02 as) as var_02,
cast(var_04 as) as var_04,
cast(var_05 as) as var_05,
cast(var_06 as) as var_06,
cast(var_07 as) as var_07,
cast(var_08 as) as var_08,
cast(var_09 as) as var_09,
cast(var_10 as) as var_10,
cast(var_11 as) as var_11,
cast(var_12 as) as var_12,
cast(var_13 as) as var_13,
cast(var_14 as) as var_14,
cast(var_15 as) as var_15,
cast(var_16 as) as var_16,
cast(var_17 as) as var_17,
cast(var_18 as) as var_18,
cast(var_19 as) as var_19,
cast(var_20 as) as var_20,
cast(var_21 as) as var_21,
cast(var_22 as) as var_22,
cast(var_23 as) as var_23,
cast(var_24 as) as var_24,
cast(var_25 as) as var_25,
cast(CEP_3_digitos as) as CEP_3_digitos,


In [7]:
lake = spark.sql(     
    """
        select
        
            cast(NUM_CPF as string) as NUM_CPF,
            try_cast(SAFRA as int) as SAFRA,
            try_cast(FLAG_INSTALACAO as int) as FLAG_INSTALACAO,
            try_cast(FPD as int) as FPD,
            cast(PROD as string) as PROD,
            cast(flag_mig2 as string) as flag_mig2,
            cast(STATUSRF as string) as STATUSRF,
            
            -- Data com tratamento para valores nulos/vazios usando to_date
            case 
                when trim(DATADENASCIMENTO) in ('null', 'NULL', '') then null
                else to_date(trim(DATADENASCIMENTO), 'dd/MM/yyyy')
            end as DATADENASCIMENTO,
            
            try_cast(var_03 as int) as var_03,
            try_cast(var_02 as int) as var_02,
            try_cast(var_04 as int) as var_04,
            try_cast(var_05 as int) as var_05,
            try_cast(var_06 as int) as var_06,
            try_cast(var_07 as int) as var_07,
            try_cast(var_08 as int) as var_08,
            try_cast(var_09 as int) as var_09,
            try_cast(var_10 as int) as var_10,
            try_cast(var_11 as int) as var_11,
            
            -- Datas com tratamento para valores nulos/vazios usando to_date
            case 
                when trim(var_12) in ('null', 'NULL', '') then null
                else to_date(trim(var_12), 'dd/MM/yyyy')
            end as var_12,
            
            case 
                when trim(var_13) in ('null', 'NULL', '') then null
                else to_date(trim(var_13), 'dd/MM/yyyy')
            end as var_13,
            
            try_cast(var_14 as int) as var_14,
            try_cast(var_15 as int) as var_15,
            try_cast(var_16 as int) as var_16,
            try_cast(var_17 as int) as var_17,
            try_cast(var_18 as int) as var_18,
            try_cast(var_19 as int) as var_19,
            cast(var_20 as string) as var_20,
            cast(var_21 as string) as var_21,
            cast(var_22 as string) as var_22,
            cast(var_23 as string) as var_23,
            cast(var_24 as string) as var_24,
            cast(var_25 as string) as var_25,
            try_cast(CEP_3_digitos as int) as CEP_3_digitos,
            
            {pdthproc} as DATPROC

        from
            raw_00
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

3900378

In [8]:
lake.show(5)

+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+----------+------+------+------+------+------+------+------+------------+------------+------+----------+--------------------+-------------+--------------+
|    NUM_CPF| SAFRA|FLAG_INSTALACAO| FPD|PROD|flag_mig2|STATUSRF|DATADENASCIMENTO|var_03|var_02|var_04|var_05|var_06|var_07|var_08|var_09|var_10|var_11|    var_12|    var_13|var_14|var_15|var_16|var_17|var_18|var_19|var_20|      var_21|      var_22|var_23|    var_24|              var_25|CEP_3_digitos|       DATPROC|
+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+----------+------+------+------+------+------+------+------+------------+------------+------+----------+--------------------+-------------+--------------+
|77789989YZZ|202503|              1|   0| CMV|

In [9]:
for col in lake.columns:
    agg_result = lake.agg(
        {col: "count"} 
    ).collect()[0]
    
    total = lake.count()
    nao_nulos = agg_result[f"count({col})"]
    nulos = total - nao_nulos
    
    if nulos > 0:
        print(f"{col}: {nulos} nulos ({nulos/total*100:.2f}%)")

FPD: 1203757 nulos (30.86%)
flag_mig2: 1266478 nulos (32.47%)
STATUSRF: 15154 nulos (0.39%)


DATADENASCIMENTO: 16831 nulos (0.43%)
var_03: 269970 nulos (6.92%)
var_02: 3685278 nulos (94.49%)
var_04: 15154 nulos (0.39%)
var_05: 196424 nulos (5.04%)
var_06: 3157126 nulos (80.94%)


var_07: 3467224 nulos (88.89%)
var_08: 3157325 nulos (80.95%)
var_09: 2223690 nulos (57.01%)
var_10: 3898192 nulos (99.94%)


var_11: 3884796 nulos (99.60%)


var_12: 1490335 nulos (38.21%)


var_13: 3459231 nulos (88.69%)
var_14: 3533608 nulos (90.60%)


var_15: 3900378 nulos (100.00%)
var_16: 3315375 nulos (85.00%)
var_17: 3315375 nulos (85.00%)


var_18: 3900378 nulos (100.00%)


var_19: 3900378 nulos (100.00%)
var_20: 3827574 nulos (98.13%)
var_21: 1490335 nulos (38.21%)
var_22: 3533608 nulos (90.60%)
var_23: 3315375 nulos (85.00%)
var_24: 1490335 nulos (38.21%)
var_25: 467469 nulos (11.99%)
CEP_3_digitos: 292051 nulos (7.49%)


In [10]:
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, SAFRA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count() 

3900378

In [11]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5)

+------+------------+-------------+
| SAFRA|total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|      653586|       653586|
|202411|      665737|       665737|
|202412|      646037|       646037|
|202501|      667227|       667227|
|202502|      619961|       619961|
+------+------------+-------------+
only showing top 5 rows



In [12]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_dados_cadastrais/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA = s.SAFRA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


Tabela silver não existe. Criando...


In [12]:
spark.stop()